In [1]:
import pandas as pd
from dlx_experiment import solve_gurobi

records = []

for z in [4, 8, 12]:
    for INS in [500, 1000, 2500, 5000]:
        for h in range(20):
            guro_integer, guro_obj, computation_time = solve_gurobi(
                data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
                instance=z,
                users=INS,
                sample=h,
            )

            records.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Solver": "Gurobi",
                "Computation Time(s)": computation_time,
                "Total Allocated Demand": guro_obj,
            })

            print(
                f"Completed: z={z}, INS={INS}, simulation={h}, "
                f"time={computation_time:.6f}s"
            )

Gurobi_SOL = pd.DataFrame(
    records,
    columns=[
        "Instance",
        "User",
        "Simulation",
        "Solver",
        "Computation Time(s)",
        "Total Allocated Demand",
    ],
)

Gurobi_SOL.to_csv("Gurobi_results.csv", index=False)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-04-22
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros (Max)
Model fingerprint: 0xb0614e2f
Model has 1188 linear objective coefficients
Variable types: 0 continuous, 1188 integer (1188 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]

Found heuristic solution: objective 948450.00000
Presolve removed 813 rows and 106 columns
Presolve time: 0.03s
Presolved: 1040 rows, 1082 columns, 13285 nonzeros
Found heuristic solution: objective 923275.00000
Variable types: 0 continuous, 1082 integer (1081 binary)

Root re

In [2]:
import timeit
import pandas as pd
from dlx_experiment import solve_cpsat

records = []

for z in [4, 8, 12]:
    for INS in [500, 1000, 2500, 5000]:
        for h in range(20):
            t_start = timeit.default_timer()
            
            integer, objective, computation_time = solve_cpsat(
                data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
                instance=z,
                users=INS,
                sample=h,
            )

            records.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Solver": "CP-SAT",
                "Computation Time(s)": computation_time,
                "Total Allocated Demand": objective,
            })

            print(
                f"Completed: z={z}, INS={INS}, simulation={h}, "
                f"time={computation_time:.6f}s"
            )

CP_SAT_SOL = pd.DataFrame(
    records,
    columns=[
        "Instance",
        "User",
        "Simulation",
        "Solver",
        "Computation Time(s)",
        "Total Allocated Demand",
    ],
)

CP_SAT_SOL.to_csv("CP_SAT_results.csv", index=False)

Completed CP-SAT: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=0, time=0.113654s
Completed CP-SAT: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=1, time=0.173907s
Completed CP-SAT: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=2, time=0.120410s
Completed CP-SAT: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=3, time=0.144452s
Completed CP-SAT: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=4, time=0.102379s
Completed CP-SAT: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=5, time=0.140075s
Completed CP-SAT: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=6, time=0.139675s
Completed CP-SAT: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=7, time=0.142305s
Completed CP-SAT: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=8, time=0.134303s
Completed CP-SAT: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=9, time=0.131871s
Completed 

In [ ]:
import pandas as pd
from dlx_experiment import extract_features_cpp

test_rows = []

for z in [4, 8, 12]:
    for INS in [500, 1000, 2500, 5000]:
        for h in range(20):
            X = extract_features_cpp(
                data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
                instance=z,
                users=INS,
                sample=h,
            )

            test_rows.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                **X,
                "Backtracking percentiles idx": back,
            })

df_ML = pd.DataFrame(test_rows)
df_ML.to_csv("DLX_ML_features.csv", index=False)

In [1]:
import pandas as pd
from lightgbm import LGBMClassifier

from dlx_experiment import extract_features_cpp, solve_ml_depth


DATA_DIR = "/Users/ryanli/Documents/GitHub/DLX_plus/Data"
DLX_RESULTS_FILE = "/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv"
OUTPUT_FILE = "/Users/ryanli/Documents/GitHub/DLX_plus/DLX_ML_results.csv"

INSTANCES = [4, 8, 12]
USER_COUNTS = [500, 1000, 2500, 5000]
SAMPLES = range(20)
THREADS = 10

FEATURES = [
    "B",
    "U",
    "num_rows",
    "density",
    "avg_degree",
    "max_degree",
    "degree_std",
    "D_mean",
    "D_std",
    "D_cv",
    "avg_row_len",
    "max_row_len",
    "row_density",
]


# ---------------------------------------------------------
# Load the best percentile labels produced by standard DLX+
# ---------------------------------------------------------

dlx_results = pd.read_csv(DLX_RESULTS_FILE)

label_column = "Backtracking percentiles idx"

dlx_labels = dlx_results[
    ["Instance", "User", "Simulation", label_column]
].copy()


# ---------------------------------------------------------
# Generate ML features in C++
# ---------------------------------------------------------

feature_records = []

for z in INSTANCES:
    for INS in USER_COUNTS:
        for h in SAMPLES:
            features = extract_features_cpp(
                data_dir=DATA_DIR,
                instance=z,
                users=INS,
                sample=h,
            )

            feature_records.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                **features,
            })

            print(
                f"Features completed: "
                f"z={z}, INS={INS}, simulation={h}"
            )

df_features = pd.DataFrame(feature_records)

df_ML = df_features.merge(
    dlx_labels,
    on=["Instance", "User", "Simulation"],
    how="inner",
    validate="one_to_one",
)

if len(df_ML) != len(df_features):
    raise ValueError(
        "Some feature rows do not have matching DLX+ labels."
    )

df_ML.to_csv(
    "/Users/ryanli/Documents/GitHub/DLX_plus/DLX_ML_features.csv",
    index=False,
)


Features completed: z=4, INS=500, simulation=0
Features completed: z=4, INS=500, simulation=1
Features completed: z=4, INS=500, simulation=2
Features completed: z=4, INS=500, simulation=3
Features completed: z=4, INS=500, simulation=4
Features completed: z=4, INS=500, simulation=5
Features completed: z=4, INS=500, simulation=6
Features completed: z=4, INS=500, simulation=7
Features completed: z=4, INS=500, simulation=8
Features completed: z=4, INS=500, simulation=9
Features completed: z=4, INS=500, simulation=10
Features completed: z=4, INS=500, simulation=11
Features completed: z=4, INS=500, simulation=12
Features completed: z=4, INS=500, simulation=13
Features completed: z=4, INS=500, simulation=14
Features completed: z=4, INS=500, simulation=15
Features completed: z=4, INS=500, simulation=16
Features completed: z=4, INS=500, simulation=17
Features completed: z=4, INS=500, simulation=18
Features completed: z=4, INS=500, simulation=19
Features completed: z=4, INS=1000, simulation=0
Fe


KeyboardInterrupt



In [3]:
# ---------------------------------------------------------
# Leave-one-problem-out training and DLX+:ML execution
# ---------------------------------------------------------
import timeit
ml_records = []

for z in INSTANCES:
    for INS in USER_COUNTS:
        for h in SAMPLES:
            test_mask = (
                (df_ML["Instance"] == z)
                & (df_ML["User"] == INS)
                & (df_ML["Simulation"] == h)
            )

            df_test = df_ML.loc[test_mask].reset_index(drop=True)
            df_train = df_ML.loc[~test_mask].reset_index(drop=True)

            if len(df_test) != 1:
                raise ValueError(
                    f"Expected one test row for "
                    f"z={z}, INS={INS}, simulation={h}; "
                    f"found {len(df_test)}."
                )

            model = LGBMClassifier(
                objective="multiclass",
                num_class=7,
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_samples=5,
                class_weight="balanced",
                verbose=-1,
                random_state=42,
            )

            model.fit(
                df_train[FEATURES],
                df_train[label_column].astype(int),
            )

            prediction_start = timeit.default_timer()

            predicted_percentile = int(
                model.predict(df_test[FEATURES])[0]
            )

            prediction_time = timeit.default_timer() - prediction_start

            # C++ performs:
            # 1. Greedy solution
            # 2. Predicted-depth calculation
            # 3. Parallel DLX+ search at only that depth
            result = solve_ml_depth(
                data_dir=DATA_DIR,
                predicted_percentile_index=predicted_percentile,
                instance=z,
                users=INS,
                sample=h,
                threads=THREADS,
            )

            computation_time = prediction_time + result["time"]

            

            ml_records.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking percentiles idx":
                    predicted_percentile,
                "Total Allocated Demand":
                    result["value"],
                "Computation Time(s)":
                    computation_time,
            })

            print(
                f"Completed: z={z}, INS={INS}, simulation={h}, "
                f"percentile={predicted_percentile}, "
                f"depth={result['backtracking_depth']}, "
                f"value={result['value']}, "
                f"time={computation_time:.6f}s"
            )


# ---------------------------------------------------------
# Save results using the standard DLX+ column format
# ---------------------------------------------------------

DLX_ML_sol = pd.DataFrame(
    ml_records,
    columns=[
        "Instance",
        "User",
        "Simulation",
        "Backtracking percentiles idx",
        "Total Allocated Demand",
        "Computation Time(s)",
    ],
)

DLX_ML_sol.to_csv(OUTPUT_FILE, index=False)

print(DLX_ML_sol)
print(f"Saved to: {OUTPUT_FILE}")

Completed: z=4, INS=500, simulation=0, percentile=1, depth=1, value=948450, time=0.003361s
Completed: z=4, INS=500, simulation=1, percentile=6, depth=13, value=965875, time=0.005813s
Completed: z=4, INS=500, simulation=2, percentile=5, depth=21, value=971325, time=0.005043s
Completed: z=4, INS=500, simulation=3, percentile=6, depth=24, value=963325, time=0.005288s
Completed: z=4, INS=500, simulation=4, percentile=6, depth=18, value=953100, time=0.006297s
Completed: z=4, INS=500, simulation=5, percentile=6, depth=21, value=960450, time=0.006426s
Completed: z=4, INS=500, simulation=6, percentile=5, depth=18, value=965650, time=0.007231s
Completed: z=4, INS=500, simulation=7, percentile=6, depth=19, value=963975, time=0.007250s
Completed: z=4, INS=500, simulation=8, percentile=5, depth=17, value=967275, time=0.004499s
Completed: z=4, INS=500, simulation=9, percentile=5, depth=20, value=968700, time=0.005887s
Completed: z=4, INS=500, simulation=10, percentile=6, depth=18, value=968200, tim

In [4]:
import pandas as pd
from dlx_experiment import solve_dlx_lp

DATA_DIR = "/Users/ryanli/Documents/GitHub/DLX_plus/Data"

records = []

for z in [4, 8, 12]:
    for INS in [500, 1000, 2500, 5000]:
        for h in range(20):
            result = solve_dlx_lp(
                data_dir=DATA_DIR,
                instance=z,
                users=INS,
                sample=h,
                threads=10,
            )

            records.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking percentiles idx":
                    result["percentile_index"],
                "Total Allocated Demand":
                    result["value"],
                "Computation Time(s)":
                    result["time"],
            })

            print(
                f"Completed: z={z}, INS={INS}, simulation={h}, "
                f"p={result['percentile_index']}, "
                f"value={result['value']}, "
                f"time={result['time']:.6f}s"
            )

DLX_LP_sol = pd.DataFrame(
    records,
    columns=[
        "Instance",
        "User",
        "Simulation",
        "Backtracking percentiles idx",
        "Total Allocated Demand",
        "Computation Time(s)",
    ],
)

DLX_LP_sol.to_csv("DLX_LP_results.csv", index=False)

Completed LP DLX+: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=0, p=6, value=978500, time=0.029798s
Completed LP DLX+: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=1, p=6, value=973500, time=0.016884s
Completed LP DLX+: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=2, p=6, value=973625, time=0.013108s
Completed LP DLX+: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=3, p=6, value=978275, time=0.013023s
Completed LP DLX+: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=4, p=6, value=974900, time=0.011487s
Completed LP DLX+: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=5, p=6, value=967075, time=0.012075s
Completed LP DLX+: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=6, p=6, value=980125, time=0.013315s
Completed LP DLX+: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=7, p=6, value=981025, time=0.013155s
Completed LP DLX+: z=4, INS=500, simulation=8
Co

### Validation

In [ ]:
import os,json
pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
with open(pairs_file, 'r') as f:
    pairs = json.load(f)
f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
allocate_data=f.read()
f.close()
allocate=eval(allocate_data)
colour={}
node={}
reverse_pos = {v: k for k, v in pos.items()}
for i in H:
    colour[reverse_pos[orig_row[i][0]]]=orig_row[i][1]
for i,j in pairs:
    if i in colour and j in colour:
        if colour[i]==colour[j]:#If there is a colour conflict
            print("Break_colour",i,j)
users=[]
for i in colour:
    users.extend(allocate[i])
if len(users)!=len(set(users)):#If there is a user conflict
    print("Break_users")

### Time Limit

In [23]:
def gurobi_SPK(rows_color, D_color, N, Cons, limit): 

    V = range(len(rows_color))

    model = gp.Model("MIP")
    model.setParam("TimeLimit", limit)

    x = model.addVars(V, vtype=gp.GRB.BINARY, name="x")

    model.addConstrs(
        gp.quicksum(x[b] for b in Cons[p]) <= 1
        for p in range(len(Cons))
    )

    model.addConstr(gp.quicksum(x[b] for b in V) <= N)

    model.setObjective(
        gp.quicksum(D_color[b] * x[b] for b in V),
        gp.GRB.MAXIMIZE
    )

    model.optimize()

    # No feasible solution found
    if model.SolCount == 0:
        return [], None, model.Runtime

    integer = [v for v in V if x[v].X > 0.5]

    if model.Status == gp.GRB.OPTIMAL:
        status = "Optimal"
    else:
        status = "Feasible"

    return integer, model.ObjVal, model.Runtime

def cpsat_SPK(rows_color,D_color,N,Cons,limit,log=True):
    
    V=range(len(rows_color))

    model = cp_model.CpModel()

    # Variables
    x = {b: model.NewBoolVar(f"x[{b}]") for b in V}
   

    # User coverage constraints
    for q_set in range(len(Cons)):
        model.Add(sum(x[b] for b in Cons[q_set]) <= 1)

    # Cardinality constraint
    model.Add(sum(x[b] for b in V) <= N)

    # Objective
    model.Maximize(sum(int(D_color[b]) * x[b] for b in V))

    # Solver
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = limit
    #solver.parameters.num_search_workers = 8
    #solver.parameters.log_search_progress = log

    start = time.time()
    status = solver.Solve(model)
    runtime = time.time() - start

    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:

        integer = [b for b in V if solver.Value(x[b]) >0.99]
        obj=solver.ObjectiveValue()
    else:
        integer = []
        obj = None
 

    return integer, obj, runtime

In [17]:
import os,json

In [19]:
import numpy as np
import pandas as pd
import timeit
import copy
import time
import itertools
import os
from joblib import Parallel, delayed
import json
import networkx as nx
from functions import *

In [21]:
percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

In [ ]:
N_max=60
n_colour=4
R=range(n_colour)
Gurobi_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
OR_TOOLS_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
DLX_best = pd.read_csv('/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv')
for z in [4,8,12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500,1000,2500,5000]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        for h in range(20): #problems     
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            D_time=DLX_best[(DLX_best['Instance'] == z) & (DLX_best['User'] == INS)&(DLX_best['Simulation'] == h)]['Computation Time(s)'].values[0]

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            Cons=incidence_transpose_fast(rows_color)

            t_start = timeit.default_timer()

            guro_integer,guro_obj,guro_solver_time=gurobi_SPK(rows_color,D_color,N_max,Cons,D_time)

            t_end = timeit.default_timer()

            Gurobi_SOL.loc[len(Gurobi_SOL)]=[z,INS,h,'Gurobi',t_end-t_start,guro_obj]

            t_start = timeit.default_timer()

            or_integer,or_obj,or_solver_time=cpsat_SPK(rows_color,D_color,N_max,Cons, D_time)

            t_end = timeit.default_timer()

            OR_TOOLS_SOL.loc[len(OR_TOOLS_SOL)]=[z,INS,h,'OR_Tools',t_end-t_start,or_obj]
            print(f"Completed: z={z}, INS={INS}, simulation={h}")


Gurobi_SOL.to_csv('Gurobi_results_timelimit.csv')
OR_TOOLS_SOL.to_csv('OR_TOOLS_results_timelimit.csv')

Set parameter Username
Academic license - for non-commercial use only - expires 2027-04-22
Set parameter TimeLimit to value 0.001829333
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[rosetta2] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros
Model fingerprint: 0x46f38d97
Variable types: 0 continuous, 1188 integer (1188 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 10 available processors)

Solution count 0

Time limit reached
Best objective -, best bound -, gap -
Completed: z=4, INS=500, simulation=0
Set parameter TimeLimit to value 0.003732334
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[rosetta2] - Darwin 2

In [7]:
dlx_results="/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv"

In [10]:
import pandas as pd

In [11]:
DLX_SOL = pd.read_csv('/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv')

In [12]:
DLX_SOL

,Instance,User,Simulation,Backtracking percentiles idx,Total Allocated Demand,Computation Time(s)
0,4,500,0,5,962750.00,0.001829
1,4,500,1,6,965875.00,0.003732
2,4,500,2,6,971325.00,0.002561
3,4,500,3,5,963325.00,0.002123
4,4,500,4,1,953100.00,0.002715
...,...,...,...,...,...,...
235,12,5000,15,6,6823216.77,0.041076
236,12,5000,16,6,6980823.21,0.058707
237,12,5000,17,6,6858499.52,0.058989
238,12,5000,18,6,6738620.34,0.044533


In [1]:
from dlx_experiment import run_exact_solvers_with_dlx_limits

gurobi_results, cpsat_results = run_exact_solvers_with_dlx_limits(
    data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
    dlx_results="/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv",
    output_dir="/Users/ryanli/Documents/GitHub/DLX_plus",
)

Set parameter Username
Academic license - for non-commercial use only - expires 2027-04-22
Set parameter TimeLimit to value 0.001829333
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  0.001829333

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros (Max)
Model fingerprint: 0xb0614e2f
Model has 1188 linear objective coefficients
Variable types: 0 continuous, 1188 integer (1188 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]

Found heuristic solution: objective 948450.00000
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 10 available processors)

Solution count 1: 948450 
No other solutions better than 0

Time limi

In [5]:
output_dir="/Users/ryanli/Documents/GitHub/DLX_plus"

In [7]:
gurobi_results.to_csv(
    f"{output_dir}/Gurobi_results_timelimit.csv",
    index=False,
)

cpsat_results.to_csv(
    f"{output_dir}/CP_SAT_results_timelimit.csv",
    index=False,
)

print("Results saved.")

Results saved.


In [1]:
from pathlib import Path

import pandas as pd

from dlx_experiment import run_all


DATA_DIR = Path("/Users/ryanli/Documents/GitHub/DLX_plus/Data")
OUTPUT_DIR = Path(
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "parallelisation_results"
)
GUROBI_FILE = Path(
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "Gurobi_results.csv"
)

INSTANCES = [4, 8, 12]
USER_COUNTS = [500, 1000, 2500, 5000]
SAMPLES = range(20)
THREAD_COUNTS = [1, 5, 10, 20]

INSTANCE_NAMES = {
    4: "Realistic",
    8: "Random",
    12: "Clustered",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# 1. Load optimal Gurobi benchmark values
# ---------------------------------------------------------

gurobi = pd.read_csv(GUROBI_FILE)

key_columns = ["Instance", "User", "Simulation"]

gurobi = gurobi[
    key_columns + ["Total Allocated Demand"]
].rename(
    columns={
        "Total Allocated Demand": "Optimal Demand"
    }
)

if gurobi.duplicated(key_columns).any():
    raise ValueError(
        "Gurobi_results.csv contains duplicate problem records."
    )


# ---------------------------------------------------------
# 2. Run DLX+ using different numbers of threads
# ---------------------------------------------------------

all_results = []

for threads in THREAD_COUNTS:
    print(f"\nRunning DLX+ with {threads} thread(s)")

    thread_output = OUTPUT_DIR / f"threads_{threads}"

    dlx_results = run_all(
        data_dir=DATA_DIR,
        output_dir=thread_output,
        instances=INSTANCES,
        users=USER_COUNTS,
        samples=SAMPLES,
        n_colour=4,
        threads=threads,
    )

    dlx_results["Threads"] = threads
    all_results.append(dlx_results)


dlx_all = pd.concat(all_results, ignore_index=True)


# ---------------------------------------------------------
# 3. Match each DLX+ result with its optimal value
# ---------------------------------------------------------

results = dlx_all.merge(
    gurobi,
    on=key_columns,
    how="left",
    validate="many_to_one",
)

if results["Optimal Demand"].isna().any():
    missing = results.loc[
        results["Optimal Demand"].isna(),
        key_columns,
    ]
    raise ValueError(
        "Missing optimal Gurobi values for:\n"
        f"{missing.to_string(index=False)}"
    )

results["Optimality Gap (%)"] = (
    (
        results["Optimal Demand"]
        - results["Total Allocated Demand"]
    )
    / results["Optimal Demand"]
    * 100
)

results["Problem Type"] = results["Instance"].map(
    INSTANCE_NAMES
)

results.to_csv(
    OUTPUT_DIR / "DLX_parallelisation_all_results.csv",
    index=False,
)


# ---------------------------------------------------------
# 4. Average over the 20 samples
# ---------------------------------------------------------

summary = (
    results.groupby(
        ["Instance", "Problem Type", "User", "Threads"],
        as_index=False,
    )["Optimality Gap (%)"]
    .mean()
)

table = summary.pivot(
    index=["Instance", "Problem Type", "User"],
    columns="Threads",
    values="Optimality Gap (%)",
)


desired_index = pd.MultiIndex.from_tuples(
    [
        (instance, INSTANCE_NAMES[instance], users)
        for instance in INSTANCES
        for users in USER_COUNTS
    ],
    names=["Instance", "Problem Type", "User"],
)

table = table.reindex(
    index=desired_index,
    columns=THREAD_COUNTS,
)


table.to_csv(
    OUTPUT_DIR / "DLX_parallelisation_average_gaps.csv"
)

print("\nAverage optimality gaps:")
print(table.round(2))


# ---------------------------------------------------------
# 5. Generate the LaTeX table
# ---------------------------------------------------------

latex_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Optimality gap of DLX$^+$ v1 across different "
    r"thread counts (in \%), averaged over 20 instances for "
    r"each problem type.}",
    r"\label{tab:dlx_parallelisation}",
    r"\begin{tabular}{lrrrr}",
    r"\toprule",
    r"& \multicolumn{4}{c}{Threads} \\",
    r"\cmidrule(lr){2-5}",
    r"INS & 1 & 5 & 10 & 20 \\",
    r"\midrule",
]

for instance_index, instance in enumerate(INSTANCES):
    name = INSTANCE_NAMES[instance]

    for users in USER_COUNTS:
        values = table.loc[(instance, name, users)]

        latex_lines.append(
            f"{name}-{users} & "
            + " & ".join(
                f"{values[threads]:.2f}"
                for threads in THREAD_COUNTS
            )
            + r" \\"
        )

    if instance_index < len(INSTANCES) - 1:
        latex_lines.append(r"\midrule")

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

latex_table = "\n".join(latex_lines)

latex_file = OUTPUT_DIR / "DLX_parallelisation_table.tex"
latex_file.write_text(latex_table)

print(f"\nLaTeX table saved to:\n{latex_file}")
print("\n" + latex_table)


Running DLX+ with 1 thread(s)
Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19
Completed: z=4, INS=1000, simulation=0
Completed: z=4, INS=1000, simulation=1
Completed: z=4, INS=1000, simulation=2
Completed: z=4, INS=1000, simulation=3
Completed: z=4, INS=1000, simulation=4
Comp

ValueError: Length of names must match number of levels in MultiIndex.

In [2]:
desired_index = pd.MultiIndex.from_tuples(
    [
        (instance, INSTANCE_NAMES[instance], users)
        for instance in INSTANCES
        for users in USER_COUNTS
    ],
    names=["Instance", "Problem Type", "User"],
)

table = table.reindex(
    index=desired_index,
    columns=THREAD_COUNTS,
)


table.to_csv(
    OUTPUT_DIR / "DLX_parallelisation_average_gaps.csv"
)

print("\nAverage optimality gaps:")
print(table.round(2))


# ---------------------------------------------------------
# 5. Generate the LaTeX table
# ---------------------------------------------------------

latex_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Optimality gap of DLX$^+$ v1 across different "
    r"thread counts (in \%), averaged over 20 instances for "
    r"each problem type.}",
    r"\label{tab:dlx_parallelisation}",
    r"\begin{tabular}{lrrrr}",
    r"\toprule",
    r"& \multicolumn{4}{c}{Threads} \\",
    r"\cmidrule(lr){2-5}",
    r"INS & 1 & 5 & 10 & 20 \\",
    r"\midrule",
]

for instance_index, instance in enumerate(INSTANCES):
    name = INSTANCE_NAMES[instance]

    for users in USER_COUNTS:
        values = table.loc[(instance, name, users)]

        latex_lines.append(
            f"{name}-{users} & "
            + " & ".join(
                f"{values[threads]:.2f}"
                for threads in THREAD_COUNTS
            )
            + r" \\"
        )

    if instance_index < len(INSTANCES) - 1:
        latex_lines.append(r"\midrule")

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

latex_table = "\n".join(latex_lines)

latex_file = OUTPUT_DIR / "DLX_parallelisation_table.tex"
latex_file.write_text(latex_table)

print(f"\nLaTeX table saved to:\n{latex_file}")
print("\n" + latex_table)


Average optimality gaps:
Threads                        1     5     10    20
Instance Problem Type User                         
4        Realistic    500    1.53  1.53  1.53  1.53
                      1000   3.09  2.85  2.81  2.81
                      2500   4.01  3.55  3.50  3.46
                      5000   5.32  4.61  4.51  4.49
8        Random       500    1.04  0.94  0.94  0.94
                      1000   0.98  0.90  0.89  0.89
                      2500   1.47  1.39  1.38  1.38
                      5000   1.25  1.12  1.11  1.11
12       Clustered    500    2.78  2.05  1.97  1.70
                      1000   4.82  2.81  2.27  2.07
                      2500   8.46  7.15  6.24  5.86
                      5000  11.14  9.53  9.26  8.95

LaTeX table saved to:
/Users/ryanli/Documents/GitHub/DLX_plus/parallelisation_results/DLX_parallelisation_table.tex

\begin{table}[ht]
\centering
\caption{Optimality gap of DLX$^+$ v1 across different thread counts (in \%), averaged over 20 inst

In [3]:
# ---------------------------------------------------------
# Calculate DLX+ speedup relative to Gurobi
# ---------------------------------------------------------

gurobi_times = pd.read_csv(GUROBI_FILE)[
    [
        "Instance",
        "User",
        "Simulation",
        "Computation Time(s)",
    ]
].rename(
    columns={
        "Computation Time(s)": "Gurobi Time(s)"
    }
)

speedup_results = dlx_all.merge(
    gurobi_times,
    on=["Instance", "User", "Simulation"],
    how="left",
    validate="many_to_one",
)

if speedup_results["Gurobi Time(s)"].isna().any():
    raise ValueError("Some instances are missing Gurobi runtimes.")

if (speedup_results["Computation Time(s)"] <= 0).any():
    raise ValueError("DLX+ computation times must be positive.")

speedup_results["Speedup"] = (
    speedup_results["Gurobi Time(s)"]
    / speedup_results["Computation Time(s)"]
)

speedup_results["Problem Type"] = (
    speedup_results["Instance"].map({
        4: "Realistic",
        8: "Random",
        12: "Cluster",
    })
)

speedup_results.to_csv(
    OUTPUT_DIR / "DLX_parallelisation_speedups_all.csv",
    index=False,
)


# Average the per-instance speedups over the 20 samples
speedup_summary = (
    speedup_results.groupby(
        ["Instance", "Problem Type", "User", "Threads"],
        as_index=False,
    )["Speedup"]
    .mean()
)

speedup_table = speedup_summary.pivot(
    index=["Instance", "Problem Type", "User"],
    columns="Threads",
    values="Speedup",
)

desired_index = pd.MultiIndex.from_tuples(
    [
        (instance, {4: "Realistic", 8: "Random", 12: "Cluster"}[instance], users)
        for instance in INSTANCES
        for users in USER_COUNTS
    ],
    names=["Instance", "Problem Type", "User"],
)

speedup_table = speedup_table.reindex(
    index=desired_index,
    columns=THREAD_COUNTS,
)

speedup_table.to_csv(
    OUTPUT_DIR / "DLX_parallelisation_average_speedups.csv"
)

print("\nAverage speedups:")
print(speedup_table.round(2))


# ---------------------------------------------------------
# Generate LaTeX table
# ---------------------------------------------------------

latex_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Speedup of DLX$^+$ v1 relative to Gurobi "
    r"across different thread counts, averaged over 20 "
    r"instances for each problem type.}",
    r"\label{tab:dlx_parallelisation_speedup}",
    r"\begin{tabular}{lrrrr}",
    r"\toprule",
    r"& \multicolumn{4}{c}{Threads} \\",
    r"\cmidrule(lr){2-5}",
    r"INS & 1 & 5 & 10 & 20 \\",
    r"\midrule",
]

table_names = {
    4: "Realistic",
    8: "Random",
    12: "Cluster",
}

for instance_index, instance in enumerate(INSTANCES):
    name = table_names[instance]

    for users in USER_COUNTS:
        values = speedup_table.loc[(instance, name, users)]

        latex_lines.append(
            f"{name}-{users} & "
            + " & ".join(
                f"{values[threads]:.2f}"
                for threads in THREAD_COUNTS
            )
            + r" \\"
        )

    if instance_index < len(INSTANCES) - 1:
        latex_lines.append(r"\midrule")

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

latex_table = "\n".join(latex_lines)

latex_file = OUTPUT_DIR / "DLX_parallelisation_speedup_table.tex"
latex_file.write_text(latex_table)

print(f"\nLaTeX table saved to:\n{latex_file}")
print("\n" + latex_table)


Average speedups:
Threads                          1        5        10      20
Instance Problem Type User                                   
4        Realistic    500     52.30    56.58    45.83   26.17
                      1000    45.49    37.61    23.13   11.44
                      2500    53.68    43.18    27.33   13.63
                      5000   190.38   176.12   114.34   57.52
8        Random       500     45.69    39.37    29.36   18.29
                      1000    33.55    31.07    20.16   10.21
                      2500    31.99    29.54    18.77   10.30
                      5000    36.36    33.56    18.98    9.74
12       Cluster      500     59.28    41.60    24.38   10.01
                      1000    98.11    63.87    39.73   16.97
                      2500   330.76   227.20   142.15   55.59
                      5000  4866.17  3032.82  2001.35  761.68

LaTeX table saved to:
/Users/ryanli/Documents/GitHub/DLX_plus/parallelisation_results/DLX_parallelisation_speedu

In [4]:
import pandas as pd

DLX_FILE = (
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "parallelisation_results/threads_10/DLX_results.csv"
)
CPSAT_FILE = (
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "CP_SAT_results.csv"
)

KEYS = ["Instance", "User", "Simulation"]

dlx = pd.read_csv(DLX_FILE)
cpsat = pd.read_csv(CPSAT_FILE)

dlx = dlx[KEYS + ["Computation Time(s)"]].rename(
    columns={"Computation Time(s)": "DLX+ Time(s)"}
)

cpsat = cpsat[KEYS + ["Computation Time(s)"]].rename(
    columns={"Computation Time(s)": "CP-SAT Time(s)"}
)

results = dlx.merge(
    cpsat,
    on=KEYS,
    how="inner",
    validate="one_to_one",
)

if len(results) != len(dlx):
    raise ValueError(
        f"Matched {len(results)} of {len(dlx)} DLX+ records."
    )

results["Speedup over CP-SAT"] = (
    results["CP-SAT Time(s)"] / results["DLX+ Time(s)"]
)

instance_names = {
    4: "Realistic",
    8: "Random",
    12: "Clustered",
}

results["Problem Type"] = results["Instance"].map(instance_names)

# Average of the 20 individual speedup ratios
summary = (
    results.groupby(
        ["Instance", "Problem Type", "User"],
        as_index=False,
    )
    .agg(
        Average_DLX_Time=("DLX+ Time(s)", "mean"),
        Average_CPSAT_Time=("CP-SAT Time(s)", "mean"),
        Average_Speedup=("Speedup over CP-SAT", "mean"),
    )
    .sort_values(["Instance", "User"])
)

summary["INS"] = (
    summary["Problem Type"]
    + "-"
    + summary["User"].astype(str)
)

summary = summary[
    [
        "INS",
        "Average_DLX_Time",
        "Average_CPSAT_Time",
        "Average_Speedup",
    ]
]

print(summary.round(2))

print(
    "\nOverall average speedup:",
    round(results["Speedup over CP-SAT"].mean(), 2),
)

print(
    "Speedup range:",
    round(summary["Average_Speedup"].min(), 2),
    "to",
    round(summary["Average_Speedup"].max(), 2),
)

summary.to_csv(
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "DLX_speedup_over_CPSAT_10_threads.csv",
    index=False,
)

               INS  Average_DLX_Time  Average_CPSAT_Time  Average_Speedup
0    Realistic-500              0.00                0.13            83.79
1   Realistic-1000              0.01                0.55            64.74
2   Realistic-2500              0.01                1.11           134.11
3   Realistic-5000              0.01                5.47           734.66
4       Random-500              0.00                0.09           100.52
5      Random-1000              0.00                0.14            56.06
6      Random-2500              0.01                0.27            45.51
7      Random-5000              0.01                0.29            44.00
8    Clustered-500              0.02                1.09            68.89
9   Clustered-1000              0.03                2.73            82.59
10  Clustered-2500              0.03               11.11           393.01
11  Clustered-5000              0.05              994.96         20200.52

Overall average speedup: 1834.03
Spee

In [5]:
import pandas as pd

DLX_FILE = (
    "/Users/ryanli/Documents/GitHub/DLX_plus/DLX_results.csv"
)
GUROBI_FILE = (
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "Gurobi_results_timelimit.csv"
)

KEYS = ["Instance", "User", "Simulation"]

dlx = pd.read_csv(DLX_FILE)
gurobi = pd.read_csv(GUROBI_FILE)

dlx = dlx[
    KEYS + ["Total Allocated Demand"]
].rename(
    columns={
        "Total Allocated Demand": "DLX+ Objective"
    }
)

gurobi = gurobi[
    KEYS + ["Total Allocated Demand"]
].rename(
    columns={
        "Total Allocated Demand": "Gurobi Objective"
    }
)

results = dlx.merge(
    gurobi,
    on=KEYS,
    how="left",
    validate="one_to_one",
)

# No incumbent returned within the time limit means objective value 0.
results["Gurobi Objective"] = (
    results["Gurobi Objective"].fillna(0)
)

if (results["DLX+ Objective"] <= 0).any():
    raise ValueError("DLX+ objective values must be positive.")

results["Gurobi Objective Relative to DLX+ (%)"] = (
    100
    * results["Gurobi Objective"]
    / results["DLX+ Objective"]
)

percentage = results[
    "Gurobi Objective Relative to DLX+ (%)"
]

print(f"Minimum: {percentage.min():.2f}%")
print(f"Maximum: {percentage.max():.2f}%")
print(f"Mean:    {percentage.mean():.2f}%")
print(f"Median:  {percentage.median():.2f}%")

results.to_csv(
    "/Users/ryanli/Documents/GitHub/DLX_plus/"
    "Gurobi_objective_relative_to_DLX.csv",
    index=False,
)

Minimum: 86.61%
Maximum: 100.00%
Mean:    98.05%
Median:  99.00%
